# Programming for Chemistry 2026/2027 @ UniMI

![logo](checkers.jpg)

## Lecture 11: Let's code a game
The purpose of this lecture is to create a simple console game exploiting what you have learned in the previous lectures. In particular:
* classes and objects
* lists and nested lists
* conditionals, loops

In addition you will be requested to code a *random* and computer player that follows a strategy of your choice.

**Disclaimer:** I've use an AI coding agent (Claude) to write the functions to display the checkers board in the Jupyter cell.

## Rules

**Checkers** (**Draughts** in UK english, **Dama** in italian) is a game in which two players take turns moving their pieces, trying to eat the opponent pieces and to conquer the opposite side.

Here we follow the american/english rules:
* Each side starts with 12 men. Black occupies squares 1-12 (top of the board). White occupies 21-32 (bottom). **Black moves first.**
* A **man** moves one square diagonally forward onto an empty square. Black moves down the board. White moves up.
* A **king** moves one square diagonally in any direction.
* **Capture**: if an enemy piece is diagonally adjacent and the square directly behind it is empty, you may jump over it, removing it from the board. Captures are **mandatory**.
* **Promotion**: a man that reaches the far row becomes a king.
* **Win**: you win when the opponent has no legal move.
* **Draw** (simplified): 80 consecutive half-moves with no capture and no man moved ends the game as a draw.

We'll not use these two rules, because they are difficult to implement!
* <s>**Multi-capture**: after a jump, if the same piece can jump again, it must keep jumping.</s>
* <s>If a promotion happens in the middle of a multi-capture chain, the move ends there.</s>

## 1. Utility functions

Write functions to:
1. Given a color (b or w), return the color of the opponed
2. Return the row a man of this color must reach to be crowned

In [ ]:
# Each cell of the board will hold a piece or it will be empty
# We use lower letters for a man, capital letters for a king
BLACK = "b"
WHITE = "w"
EMPTY = "."

In [ ]:
# Return the color of the opponent
def opponent(color):
    assert ...
    ...

In [ ]:
# Return the row (0 or 7) a man of this color must reach to be crowned
def king_row(color):
    assert ...
    ...


![checker_numbering](checkers_numbering.png)
Create a *dictonary* `NUM_TO_SQUARE` mapping the 1..32 squares to tuples of (row,col). Create a *dictionary* `SQUARE_TO_NUM` that maps a tuple (r,c) to the square 1..32.

In [ ]:
# Map between the 1..32 square numbers and (row, col) coordinates. Use only the dark squares.
NUM_TO_SQUARE, SQUARE_TO_NUM = {}, {}

num = 1
for ...
    for ...

In [ ]:
# test if the two dictionaries are correct
print(NUM_TO_SQUARE[7])           # (1,4)
print(SQUARE_TO_NUM[(1,2)])       # 6
print(SQUARE_TO_NUM[(2,4)])       # KeyError!

Write functions to:
1. Return a list of diagonal directions (d_row, d_cols) in which the piece can move. Take care of men and kings
2. Check if row and column are inside the board (i.e. between 0 and 7)

In [ ]:
# Return a list of tuples (delta_row, delta_col) this piece may move.
# Kings can move in both directions
def directions(piece):
    ...

In [ ]:
# check if c and r are both inside the board, i.e. between 0 and 7    
def is_inside(r, c):
    return ...

In [ ]:
# print a move from square to square with the following convention:
# square1-square2 if it is a move
# square1xsquare2 if it is a capture jump
def format_move(move):
    if abs(move[1][0] - move[0][0]) == 2:
        sep = 'x'
    else:
        sep = "-" 
    return sep.join(str(SQUARE_TO_NUM[sq]) for sq in move)

## 2. The Board class
Holds the position and knows the rules: legal moves and move execution.

The board never changes in place when a move is played: apply() returns
a NEW board. This is what makes the minimax search simple, because we
can explore a move without having to undo it.

In [ ]:
class Board:
    
    # Create the Board object. If grid is not given, initialize the positions
    def __init__(self, grid=None):
        if grid is None:
            self.initialize_grid()
        else:
            self.grid = grid
            
    # Initialize the pieces at the starting positions     
    def initialize_grid(self):
        # initialize to the value EMPTY a list of lists of size 8x8
        ...

        # fill the initial pieces: black pieces in 1-12, white pieces in 21-32
        ...

    # Return a copy of the Board, by copyiing self.grid        
    def copy(self):
        newgrid = [row.copy() for row in self.grid]
        return Board(newgrid)

    
    # Return the number of men and kings) for the given color
    def count_pieces(self, color):
        men, kings = 0, 0
        ...
        return men, kings


    # Return a list of all possible moves for the given color. If there is at
    # least one capture jump, return only the list of jumps (not steps) 
    def legal_moves(self, color):
        jumps, steps = [], []

        for r in range(8):
            for c in range(8):
                piece = self.grid[r][c]
                if piece == color or piece == color.upper():
                    jumps.extend(self.find_jump_moves(r, c))
                    steps.extend(self.find_step_moves(r, c))

        # mandatory jump rule
        if len(jumps) > 0:
            return jumps
        return steps


    # Return a list of step move from position (r,c), by moving in the directions
    # given by the directions() function. The final position must be empty.
    # Append the move as a tuple like ((r,c), (new_r,new_c)) to the list of moves.
    def find_step_moves(self, r, c):
        piece = self.grid[r][c]
        ...
        return moves

    
    # Return a list of step jump from position (r,c), by moving in the directions
    # given by the directions() function. The middle position must be of the opposite
    # color. The final position must be empty.
    # Append the move as a tuple like ((r,c), (new_r,new_c)) to the list of moves.
    def find_jump_moves(self, r, c):
        piece = self.grid[r][c]
        victim = opponent(piece.lower())
        ...
        return moves        
                

    # Execute a move: always return a new Board, the original is untouched.
    # If the move is a jump, remove the captured piece. Check for promotion moves.
    def apply(self, move):
        new_board = self.copy()
        grid = new_board.grid

        # starting and end position
        (r1, c1) = move[0]
        (r2, c2) = move[1]
        piece = grid[r1][c1]

        # move the piece away from (r1,c1)
        ...
        
        # if it is a jump capture, remove the piece in the middle
        ...
        
        # check if promotion
        ...
        
        # place the piece in (r2,c2)
        grid[r2][c2] = piece
        
        return new_board
    
    
    # Return a string representation of the board (DO NOT MODIFY)
    def __str__(self):
        """Text view: pieces as b/B/w/W, empty dark squares show their number."""
        lines = []
        for r in range(8):
            cells = []
            for c in range(8):
                if (r + c) % 2 == 0:
                    cells.append("   ")
                elif self.grid[r][c] == EMPTY:
                    cells.append(f"{SQUARE_TO_NUM[(r, c)]:>2} ")
                else:
                    cells.append(f" {self.grid[r][c]} ")
            lines.append("".join(cells))
        return "\n".join(lines)

    
    # Return an SVG image of the board (DO NOT MODIFY)
    def _repr_svg_(self, s=60):
        """Jupyter renders this automatically when a Board is the last
        expression of a cell."""
        N = 8
        size = N * s
        parts = []
        for r in range(N):
            for c in range(N):
                color = "#f0d9b5" if (r + c) % 2 == 0 else "#b58863"
                parts.append(f'<rect x="{c*s}" y="{r*s}" width="{s}" height="{s}" fill="{color}"/>')
        fills = {BLACK: ("#2b2b2b", "#555555"), WHITE: ("#f5f5f5", "#bbbbbb")}
        for r in range(N):
            for c in range(N):
                p = self.grid[r][c]
                if p == EMPTY:
                    continue
                fill, ring = fills[p.lower()]
                cx, cy, rad = c * s + s / 2, r * s + s / 2, s * 0.38
                parts.append(f'<circle cx="{cx+2}" cy="{cy+3}" r="{rad}" fill="rgba(0,0,0,0.35)"/>')
                parts.append(f'<circle cx="{cx}" cy="{cy}" r="{rad}" fill="{fill}" stroke="{ring}" stroke-width="2"/>')
                parts.append(f'<circle cx="{cx}" cy="{cy}" r="{rad*0.7}" fill="none" stroke="{ring}" stroke-width="2"/>')
                if p.isupper():
                    parts.append(f'<circle cx="{cx}" cy="{cy}" r="{rad*0.35}" fill="gold" stroke="#8a6d00" stroke-width="2"/>')
        for (r, c), num in SQUARE_TO_NUM.items():
            parts.append(f'<text x="{c*s + s*0.07}" y="{r*s + s*0.07}" font-family="Arial, sans-serif" '
                         f'font-size="{s*0.2}" fill="#f5e9d0" dominant-baseline="hanging">{num}</text>')
        return (f'<svg xmlns="http://www.w3.org/2000/svg" width="{size}" height="{size}" '
                f'viewBox="0 0 {size} {size}">' + "".join(parts) + "</svg>")


    # Draw or print the board: if possible as an SVG image in Jupyer, text elsewhere
    # (DO NOT MODIFY)
    def show_board(self, s=60): 
        try:
            from IPython.display import SVG, display, clear_output
            clear_output(wait=True)
            display(SVG(self._repr_svg_(s)))
        except ImportError:
            print(self, "\n")

### 2.1 Let's test some of the methods we have implemented.

In [ ]:
# Let's start with the initial board
b = Board()
b.show_board()
print(b.count_pieces('b'), b.count_pieces('w'))

In [ ]:
# let's try a move
moves = b.legal_moves(WHITE)
print("white legal moves:", [format_move(m) for m in moves])

In [ ]:
move = moves[2]
b = b.apply(move)
b.show_board()

In [ ]:
move = (NUM_TO_SQUARE[18], NUM_TO_SQUARE[14])
b = b.apply(move)
b.show_board()

In [ ]:
# let's try a jump capture
moves = b.legal_moves(BLACK)
print("black legal moves:", [format_move(m) for m in moves])

In [ ]:
move = moves[1]
b = b.apply(move)
b.show_board()

## 3. Players

We now create an **abstract** base class `Player` that takes the color and the name of the player. Every class derived from `AbstractPlayer` must implement the `choose_move()` method.

In [ ]:
from abc import ABC, abstractmethod

class AbstractPlayer(ABC):
    def __init__(self, color, name):
        self.color = color
        self.name = name

    @abstractmethod
    def choose_move(self, board):
        # Return a move (tuple of squares) chosen from board.legal_moves(self.color)
        pass

### 3.1 Create a `HumanPlayer` class derived from `AbstractPlayer` that print all legal moves and let you choose one, using the `input()` function.

In [ ]:
class HumanPlayer(AbstractPlayer):
    
    def choose_move(self, board):
        ...

### 3.2 Create a `RandomComputerPlayer` class derived from `AbstractPlayer` will choose a random move from the list of legal moves
Hint: use `random.choice()`

In [ ]:
import random

class RandomComputerPlayer(AbstractPlayer):
    
    def choose_move(self, board):
        ...

## 4. Let's write the function that plays the game
Create a `Board` object. The black player begins. Make an infinite loop, taking turns.
* If you don't have legal moves, your opponent wins.
* If there is no capture for 80 moves, it's a draw.
* If no men moves for 80 moves, it's a draw.

Call `board.show_board()` if the `show` argument is True. Print also the number of men and kings of both players.

In [ ]:
def play_checkers(black, white, show=True):
    """Play a full game and return BLACK, WHITE, or None for a draw."""
    board = Board()
    players = {BLACK: black, WHITE: white}
    max_quiet_plies = 80
    
    # black moves firts
    turn = BLACK
    quiet = 0
    
    # show the initial situation
    if show:
        board.show_board()

    # game loop
    while True:
        # get the legal moves of one color
        moves = board.legal_moves(turn)
        
        # no valid moves: you opponent wins        
        if len(moves) == 0:                          
            winner = opponent(turn)
            break

        # nothing is hanppening: draw
        if  quiet >= max_quiet_plies:
            winner = None
            break

        # ask the player for a move
        move = players[turn].choose_move(board)

        # Progress counter for the draw rule: reset on captures and man moves.
        there_was_capture = abs(move[1][0] - move[0][0]) == 2
        was_man_move = board.grid[move[0][0]][move[0][1]].islower()
        if (there_was_capture or was_man_move): 
            quiet = 0
        else:
            quiet += 1

        # perform the actual move
        board = board.apply(move)

        # print the board and number of men, kings left
        if show:
            board.show_board()
            print(f"{players[turn].name} played {format_move(move)}")

            black_men, black_kings = board.count_pieces(BLACK)
            white_men, white_kings = board.count_pieces(WHITE)
            print(f'Black has {black_men} men and {black_kings} kings. White has {white_men} men and {white_kings} kings')

        # the turn passes to the opponent
        turn = opponent(turn)
     
    # end of game
    if show:
        if winner is None:
            print("Draw.")
        else:
            print(f"{players[winner].name} wins!")

    return winner

### 3.1: Let's try human vs human

In [ ]:
black = HumanPlayer(BLACK, "Nero")
white = HumanPlayer(WHITE, "Bianco")
winner = play_checkers(black, white)

### 3.2: Let's try human vs computer

In [ ]:
black = HumanPlayer(BLACK, "Nero")
white = RandomComputerPlayer(WHITE, "Bianco")
winner = play_checkers(black, white)

### 3.3: Let's try computer vs computer

In [ ]:
black = RandomComputerPlayer(BLACK, "Nero")
white = RandomComputerPlayer(WHITE, "Bianco")

wins = []
for _ in range(100):
    wins.append(play_checkers(black, white, show=False))
    
print("Black wins:", wins.count(BLACK))
print("White wins:", wins.count(WHITE))
print("Draw      :", wins.count(None))

## 4. AIPlayer using the mini-max algorithm
This code has been generated by Claude AI. **It is unbeatable!**

It is anyway instructive to read it and try to understand how it works. It is very generic and it can be adapted to other turn games, provided that you code the appropriate `_evaluate()` method.

In [ ]:
import math

class AIPlayer(AbstractPlayer):
    """Computer player using minimax search with alpha-beta pruning.

    IDEA OF MINIMAX
    ---------------
    We look `depth` half-moves ahead. At the leaves of the search tree the
    position is scored by a heuristic (_evaluate): positive is good for the
    AI, negative is good for the opponent. Then scores travel up the tree:
      * on the AI's turn we assume it picks the move with the MAX score,
      * on the opponent's turn we assume it picks the move with the MIN score
        (i.e. the opponent plays its best reply).
    The AI plays the root move with the best guaranteed outcome.

    ALPHA-BETA PRUNING
    ------------------
    alpha = best score the maximizer can already guarantee,
    beta  = best score the minimizer can already guarantee.
    If alpha >= beta, the rest of that branch cannot influence the final
    decision and is skipped. The result is identical to plain minimax, only
    faster.
    """

    WIN = 10_000            # score of a won position
    MAN, KING = 100, 160    # material values
    ADVANCE = 3             # bonus per row a man has advanced
    MAX_EXTENSION = 4       # extra plies allowed to finish forced captures

    def __init__(self, color, name, depth=5):
        super().__init__(color, name)
        self.depth = depth

        
    def choose_move(self, board):
        moves = board.legal_moves(self.color)
        if len(moves) == 1:
            return moves[0]                          # forced move, no need to think
        best_score, best_moves = -math.inf, []
        for move in moves:
            # Full alpha-beta window for each root move so that equal scores
            # are real ties (we then pick randomly among them for variety).
            score = self._minimax(board.apply(move), self.depth - 1,
                                  -math.inf, math.inf, opponent(self.color))
            if score > best_score:
                best_score, best_moves = score, [move]
            elif score == best_score:
                best_moves.append(move)
        return random.choice(best_moves)

    
    def _minimax(self, board, depth, alpha, beta, turn):
        moves = board.legal_moves(turn)

        # Terminal position: the side to move has no moves, so it has lost.
        # Adding `depth` makes the AI prefer quick wins and slow defeats.
        if not moves:
            return -(self.WIN + depth) if turn == self.color else self.WIN + depth

        # Depth limit reached. If the position is in the middle of a capture
        # exchange (captures are forced), keep searching a few extra plies;
        # otherwise the static score would misjudge it ("horizon effect").
        if depth <= 0:
            in_exchange = abs(moves[0][1][0] - moves[0][0][0]) == 2
            if not in_exchange or depth <= -self.MAX_EXTENSION:
                return self._evaluate(board)

        if turn == self.color:                       # maximizing player (the AI)
            value = -math.inf
            for move in moves:
                value = max(value, self._minimax(board.apply(move), depth - 1,
                                                 alpha, beta, opponent(turn)))
                alpha = max(alpha, value)
                if alpha >= beta:                    # opponent would never allow this line
                    break
            return value
        else:                                        # minimizing player (the opponent)
            value = math.inf
            for move in moves:
                value = min(value, self._minimax(board.apply(move), depth - 1,
                                                 alpha, beta, opponent(turn)))
                beta = min(beta, value)
                if alpha >= beta:
                    break
            return value

        
    def _evaluate(self, board):
        """Heuristic score from the AI's point of view.

        Material dominates (a king is worth more than a man); men also get a
        small bonus for advancing, since they are closer to being crowned.
        """
        score = 0
        for r in range(8):
            for c in range(8):
                p = board.grid[r][c]
                if p == EMPTY:
                    continue
                color = p.lower()
                if p.isupper():
                    value = self.KING
                else:
                    advanced = r if color == BLACK else 7 - r
                    value = self.MAN + self.ADVANCE * advanced
                score += value if color == self.color else -value
                
        return score

In [ ]:
from tqdm import tqdm

black = AIPlayer(BLACK, "Black", depth=3)
white = AIPlayer(WHITE, "White", depth=2)

wins = []
for i in tqdm(range(100)):
    wins.append(play_checkers(black, white, show=False))

print("Black wins:", wins.count(BLACK))
print("White wins:", wins.count(WHITE))
print("Draw      :", wins.count(None))